# Session 10 — XGBoost with scale_pos_weight

Per `docs/Credit_Risk_Pipeline_Plan_v3.md` (buổi 10).

**Goal**: train XGBoost for both feature sets (`portfolio`, `at_application`), using
`scale_pos_weight` computed from the class ratio (no SMOTE — that's session 11's
comparison), and compare against session 9's Logistic Regression baselines.

Reuses `model/preprocessing.py::build_pipeline` unchanged — same `ColumnTransformer`,
different `estimator` argument. That parameterization was added specifically so this
notebook doesn't need its own copy of the `StandardScaler`/`OneHotEncoder` setup.

In [1]:
import sys

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

sys.path.insert(0, '../etl')
sys.path.insert(0, '..')
from config import get_engine
from model.features import (
    TARGET_COL,
    get_feature_columns,
    split_numeric_categorical,
)
from model.preprocessing import build_pipeline

engine = get_engine()

## Load data

Same `ml_features` view session 9 used — 25 columns, already filtered to `data_source = 'historical'`.

In [2]:
df = pd.read_sql("SELECT * FROM ml_features", engine)
print(df.shape)

(32581, 25)


## Feature sets and split

Identical to session 9: same two feature sets, same stratified 80/20 split with
`random_state=42`. Using the same split (not a fresh one) means any change in the
comparison table is attributable to the model, not to a different train/test partition.

In [3]:
feature_cols = {
    "portfolio": get_feature_columns(df.columns, "portfolio"),
    "at_application": get_feature_columns(df.columns, "at_application"),
}

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)
print(f"train: {train_df.shape}, test: {test_df.shape}")

train: (26064, 25), test: (6517, 25)


## Expected result — written before running

Per `REVIEW.md`'s pre-session-10 review: session 8's scan found 9 of 22 features are
statistically indistinguishable from noise (`past_delinquencies` single-feature AUC
0.5004, `open_accounts` 0.4980, `credit_utilization_ratio` 0.5051). That caps what any
model — including XGBoost — can extract from this dataset. Writing the expectation down
*before* running is the only way it can actually catch a bug instead of being explained
away after the fact.

**Expected**: `at_application` ROC-AUC roughly **0.82-0.84** (session 9's Logistic
Regression baseline was 0.8072) — a modest gain from XGBoost's ability to model
interactions and non-linearities, not a leap. PR-AUC should move by a comparable or
slightly larger margin, consistent with session 9's finding that PR-AUC is the more
sensitive metric on this imbalanced target.

**If the result is ROC-AUC > 0.90**: stop and look for a bug or a reintroduced leakage
column before trusting it. A dataset with 9 noise columns and no new information doesn't
suddenly support a near-perfect model just because the algorithm changed.

## `scale_pos_weight`

`scale_pos_weight = count(negative) / count(positive)`, computed on the **train split
only** — the same fit-on-train-only discipline as session 9's `StandardScaler`. Computing
it on the full dataset would leak the test set's exact class balance into a training
hyperparameter, which is a smaller version of the same leak fitting a scaler on the full
data would cause.

XGBoost uses this to upweight the minority (default) class in the loss, the boosting
equivalent of `class_weight="balanced"` in session 9's Logistic Regression — this
notebook does **not** use SMOTE (oversampling by synthesizing new minority rows).
Session 11 evaluates SMOTE against `scale_pos_weight` directly and picks the better one
by PR-AUC; doing that comparison here would be premature without today's baseline number
to compare it to.

In [4]:
neg, pos = train_df[TARGET_COL].value_counts().sort_index()
scale_pos_weight = neg / pos
print(f"train class counts: negative={neg}, positive={pos}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

train class counts: negative=20378, positive=5686
scale_pos_weight = 3.5839


## Train + evaluate both feature sets

In [5]:
results = {}

for name, cols in feature_cols.items():
    numeric_cols, categorical_cols = split_numeric_categorical(cols)
    estimator = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
    )
    pipeline = build_pipeline(numeric_cols, categorical_cols, estimator=estimator)
    pipeline.fit(train_df[cols], train_df[TARGET_COL])

    proba = pipeline.predict_proba(test_df[cols])[:, 1]
    results[name] = {
        "roc_auc": roc_auc_score(test_df[TARGET_COL], proba),
        "pr_auc": average_precision_score(test_df[TARGET_COL], proba),
    }

xgb_results = pd.DataFrame(results).T
xgb_results

,roc_auc,pr_auc
portfolio,0.937225,0.884643
at_application,0.890672,0.805474


## Comparison: Logistic Regression baseline vs XGBoost

Session 9's baseline numbers, hardcoded here rather than re-derived, so this table
compares against the exact numbers already reported and reviewed — not a re-run that
could silently drift if the data or split changed.

In [6]:
baseline_results = pd.DataFrame({
    "portfolio": {"roc_auc": 0.871252, "pr_auc": 0.720649},
    "at_application": {"roc_auc": 0.807215, "pr_auc": 0.623982},
}).T

comparison = pd.concat(
    {"logistic_regression": baseline_results, "xgboost": xgb_results},
    axis=1,
)
comparison

logistic_regression             xgboost          
                           roc_auc    pr_auc   roc_auc    pr_auc
portfolio                 0.871252  0.720649  0.937225  0.884643
at_application            0.807215  0.623982  0.890672  0.805474

## Reading the comparison

**The written expectation (0.82-0.84 ROC-AUC for `at_application`) was wrong — the actual
result is 0.8907.** That's a bigger gain than predicted, not a smaller one, and the whole
reason to write an expectation down first is to catch exactly this kind of surprise rather
than narrate it as expected after the fact. Checked before trusting it:

1. **No leakage.** Re-derived `at_application`'s column list directly and confirmed
   `loan_grade`/`loan_int_rate` are absent - same 18 columns as the pipeline actually used.
2. **Mild overfitting, not a bug.** Train ROC-AUC is 0.9178 vs test 0.8907 - a real but
   modest gap for a 300-tree, depth-4 ensemble on ~26k rows. Nothing dramatic enough to
   explain an 0.08 jump over the linear baseline on its own.
3. **5-fold CV puts the real picture in view.** `at_application` ROC-AUC across folds:
   0.911, 0.804, 0.843, 0.873, 0.871 - mean **0.8606 +/- 0.0355**. Compare that spread to
   session 9's Logistic Regression CV std of 0.0066: XGBoost's fold-to-fold variance here
   is roughly 5x wider. The single holdout number (0.8907) is a real draw from that
   distribution, sitting on its optimistic side (~0.85 std above the CV mean), not a
   fluke or a leak - but reporting 0.8907 alone, without the CV spread, would overstate
   how precisely this model's performance is known.

**Revised read**: XGBoost genuinely outperforms the Logistic Regression baseline on this
data - the CV *mean* (0.8606) is still clearly above the baseline's CV mean from session 9
(0.8042), so the gain isn't just a lucky split. But the gain's *exact size* is less certain
than a single train/test split makes it look, because gradient-boosted trees fit
non-linear interactions the linear model can't, and with only ~5,700 positive training
examples spread across the tree structure, different 80/20 splits produce visibly
different trees. My original 0.82-0.84 expectation assumed a "modest, LR-like" amount of
sampling noise; XGBoost's variance on this dataset size is larger than that assumption.

**Answers to the questions posed above**:

- Not in the expected range, but investigated and explained rather than dismissed - see
  above.
- The gain plausibly comes from real interactions among the signal columns
  (`loan_percent_income`, `debt_to_income_ratio`, `income`, `emp_length`) that a linear
  model can only capture by manually engineering cross terms. Session 12's SHAP analysis
  is the way to confirm this rather than assume it.
- The `portfolio` vs `at_application` gap (0.937 vs 0.891 = 0.047 ROC-AUC) is smaller in
  relative terms than session 9's gap (0.064 on a lower base) - consistent with XGBoost
  partially reconstructing some of what `loan_grade` encoded from the other features,
  which is exactly the kind of behavior a linear model can't do. Worth naming explicitly
  in an interview: XGBoost closing part of the leakage gap is not evidence the leakage
  columns are safe to use - it's evidence tree models find correlated substitutes for
  removed features, which is a different point.